# Day 2: Defect Configuration Analysis

This notebook extends the Day 1 lattice model by studying controlled hydrogen-like defect configurations.


## Mathematical Formulas Used in This Notebook

The clean one-dimensional tight-binding Hamiltonian is

$$
H_0 = \sum_{i=1}^{N} \epsilon_0 |i\rangle\langle i|
- t\sum_{i=1}^{N-1}\left(|i\rangle\langle i+1| + |i+1\rangle\langle i|\right),
$$

where $\epsilon_0$ is the on-site energy and $t$ is the hopping strength.

Hydrogen-like defects are modeled as local diagonal perturbations:

$$
H = H_0 + \sum_{d \in D} V_d |d\rangle\langle d|.
$$

The spectrum is obtained from

$$
H\mathbf{c}^{(n)} = E_n\mathbf{c}^{(n)}.
$$

The spectral shift caused by defects is

$$
\Delta E_n = E_n^{\mathrm{defected}} - E_n^{\mathrm{clean}}.
$$

Localization is measured using the inverse participation ratio:

$$
\mathrm{IPR}_n = \sum_{i=1}^{N}|c_i^{(n)}|^4.
$$

The average localization is

$$
\overline{\mathrm{IPR}} = \frac{1}{N}\sum_{n=1}^{N}\mathrm{IPR}_n.
$$

The transport score is represented as

$$
T = \frac{1}{1+\overline{\mathrm{IPR}}}.
$$

This means that configurations producing stronger localization are expected to have lower transport scores.


## How to read this notebook

This notebook compares different physical choices for hydrogen-like defects.  
The goal is not only to generate numbers, but to understand **which defect properties control localization and transport**.

You will study three questions:

1. What changes when a clean lattice becomes defected?
2. What happens when the number or strength of defects increases?
3. Does the spatial arrangement of defects matter?

This notebook is important because it connects the mathematical model to physical intuition.


## Scientific Question

How sensitive are localization and transport descriptors to hydrogen defect configurations?

Day 2 studies defect number, defect strength, and defect arrangement.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.linalg import eigh

ROOT = Path('..').resolve()
DATA_DIR = ROOT / 'data'
FIGURE_DIR = ROOT / 'figures'

DATA_DIR.mkdir(exist_ok=True)
FIGURE_DIR.mkdir(exist_ok=True)

### Explanation of the code cell above

This cell imports the libraries and prepares folders.

Compared with Notebook 1, this notebook also imports `eigh` from `scipy.linalg`. This function is used to solve the eigenvalue problem for symmetric/Hermitian matrices.

The folders `data` and `figures` are created so that all numerical outputs and plots are stored in a consistent project structure.


## Day 1 Helper Functions

In [ ]:
def build_clean_hamiltonian(n_sites=40, hopping=1.0, epsilon_0=0.0):
    """Build a clean 1D nearest-neighbor tight-binding Hamiltonian."""
    hamiltonian = np.zeros((n_sites, n_sites), dtype=float)
    np.fill_diagonal(hamiltonian, epsilon_0)

    for i in range(n_sites - 1):
        hamiltonian[i, i + 1] = -hopping
        hamiltonian[i + 1, i] = -hopping

    return hamiltonian


def compute_ipr(eigenvectors):
    """Compute inverse participation ratio for each eigenstate."""
    return np.sum(np.abs(eigenvectors) ** 4, axis=0)


def add_defects_at_sites(hamiltonian, defect_sites, defect_strength=2.0):
    """Add hydrogen-like defects to specific lattice sites."""
    defected = hamiltonian.copy()
    for site in defect_sites:
        defected[int(site), int(site)] += defect_strength
    return defected


def analyze_hamiltonian(hamiltonian):
    """Compute spectrum, eigenvectors, IPR, and a transport score."""
    eigenvalues, eigenvectors = eigh(hamiltonian)
    ipr_values = compute_ipr(eigenvectors)
    mean_ipr = float(np.mean(ipr_values))
    max_ipr = float(np.max(ipr_values))

    return {
        'eigenvalues': eigenvalues,
        'eigenvectors': eigenvectors,
        'ipr_values': ipr_values,
        'mean_ipr': mean_ipr,
        'max_ipr': max_ipr,
        'transport_score': float(1.0 / mean_ipr),
    }

### Explanation of the code cell above

This cell defines reusable physics functions.

- `build_clean_hamiltonian` creates a one-dimensional lattice without defects.
- `compute_ipr` measures how localized each eigenstate is.
- `add_defects_at_sites` modifies selected diagonal entries of the Hamiltonian to represent local hydrogen-like perturbations.
- `analyze_hamiltonian` solves the Hamiltonian and returns the main descriptors: eigenvalues, eigenvectors, IPR, mean IPR, max IPR, and transport score.

The purpose of this cell is to avoid repeating the same physics operations in every experiment.


## Basic Parameters

In [ ]:
n_sites = 40
hopping = 1.0
epsilon_0 = 0.0
defect_strength = 2.0

H_clean = build_clean_hamiltonian(n_sites, hopping, epsilon_0)
clean_results = analyze_hamiltonian(H_clean)

### Explanation of the code cell above

This cell defines the clean reference system.

A clean lattice has no hydrogen-like defects. It is important because all defected systems can be compared against this baseline.

The variables mean:

- `n_sites`: number of lattice sites,
- `hopping`: coupling between neighboring sites,
- `epsilon_0`: baseline on-site energy,
- `defect_strength`: strength that will later be added to selected defect sites.

`clean_results` stores the spectrum and localization descriptors of the clean lattice.


## Part A: Clean Versus Defected Spectrum

In [ ]:
defect_sites = [10, 20, 30]
H_defected = add_defects_at_sites(
    H_clean,
    defect_sites=defect_sites,
    defect_strength=defect_strength,
)
defected_results = analyze_hamiltonian(H_defected)

plt.figure(figsize=(8, 5))
plt.plot(clean_results['eigenvalues'], marker='o', linestyle='none', label='Clean lattice')
plt.plot(defected_results['eigenvalues'], marker='x', linestyle='none', label='Defected lattice')
plt.xlabel('Eigenstate index')
plt.ylabel('Energy')
plt.title('Clean vs Defected Energy Spectrum')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'clean_vs_defected_spectrum_day2.png', dpi=200)
plt.show()

### Mathematical meaning of this comparison cell

This cell compares the clean Hamiltonian $H_0$ with the defected Hamiltonian

$$
H = H_0 + \sum_{d\in D}V_d|d\rangle\langle d|.
$$

The plotted eigenvalues are the solutions of

$$
H_0\mathbf{c}^{(n)}_0 = E_n^{\mathrm{clean}}\mathbf{c}^{(n)}_0,
\qquad
H\mathbf{c}^{(n)} = E_n^{\mathrm{defected}}\mathbf{c}^{(n)}.
$$


### Explanation of the code cell above

This cell adds a controlled set of defects to the clean lattice.

The selected defect sites are `[10, 20, 30]`. After adding these local perturbations, the notebook analyzes the new Hamiltonian and compares its energy spectrum with the clean spectrum.

The plot helps show how defects shift the allowed energy states. If the defected spectrum differs strongly from the clean spectrum, the defects are significantly modifying the electronic/transport structure of the model.


## Spectral Shift

The spectral shift measures how much each energy level changes after defects are added.

In [ ]:
delta_E = defected_results['eigenvalues'] - clean_results['eigenvalues']

plt.figure(figsize=(8, 5))
plt.plot(delta_E, marker='o')
plt.xlabel('Eigenstate index')
plt.ylabel('Delta E_n')
plt.title('Spectral Shift Caused by Hydrogen-Like Defects')
plt.grid(True)
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'spectral_shift_day2.png', dpi=200)
plt.show()

### Mathematical meaning of the spectral-shift cell

The spectral shift is calculated as

$$
\Delta E_n = E_n^{\mathrm{defected}} - E_n^{\mathrm{clean}}.
$$

Large values of $\Delta E_n$ indicate that the corresponding eigenstate is strongly affected by the defect potential.


### Explanation of the code cell above

This cell computes the spectral shift caused by defects.

`delta_E` is calculated as:

defected eigenvalues minus clean eigenvalues.

This tells us how much each energy level changes after introducing hydrogen-like defects. The plot is useful because some eigenstates may be more sensitive to defects than others.

Large positive or negative shifts indicate that the corresponding energy states are strongly affected by the defect configuration.


## Part B: Defect Number Sweep

In [ ]:
def sweep_defect_number(
    n_sites=40,
    hopping=1.0,
    epsilon_0=0.0,
    defect_strength=2.0,
    max_defects=15,
    seed=42,
):
    rng = np.random.default_rng(seed)
    clean = build_clean_hamiltonian(n_sites, hopping, epsilon_0)
    rows = []

    for num_defects in range(1, max_defects + 1):
        sites = np.sort(rng.choice(n_sites, size=num_defects, replace=False))
        defected = add_defects_at_sites(clean, sites, defect_strength)
        results = analyze_hamiltonian(defected)
        rows.append({
            'experiment': 'defect_number_sweep',
            'arrangement': 'random',
            'num_defects': num_defects,
            'defect_strength': defect_strength,
            'defect_sites': ';'.join(map(str, sites.tolist())),
            'mean_ipr': results['mean_ipr'],
            'max_ipr': results['max_ipr'],
            'transport_score': results['transport_score'],
        })

    return pd.DataFrame(rows)


df_number = sweep_defect_number()
df_number.head()

### Explanation of the code cell above

This cell defines and runs a sweep over the number of defects.

For each number of defects from `1` to `max_defects`, the function:

1. randomly selects defect sites,
2. adds defects to the clean Hamiltonian,
3. analyzes the defected Hamiltonian,
4. stores the localization and transport descriptors.

The resulting table `df_number` allows us to study whether increasing the number of defects systematically changes localization and transport.


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(df_number['num_defects'], df_number['mean_ipr'], marker='o')
plt.xlabel('Number of defects')
plt.ylabel('Mean IPR')
plt.title('Effect of Defect Number on Localization')
plt.grid(True)
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'defect_number_vs_mean_ipr_day2.png', dpi=200)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(df_number['num_defects'], df_number['transport_score'], marker='o')
plt.xlabel('Number of defects')
plt.ylabel('Transport score')
plt.title('Effect of Defect Number on Transport Descriptor')
plt.grid(True)
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'defect_number_vs_transport_day2.png', dpi=200)
plt.show()

### Explanation of the code cell above

This cell visualizes the defect-number sweep.

The first plot shows how mean IPR changes as the number of defects increases. A rising trend would suggest stronger localization.

The second plot shows how the transport score changes with the number of defects. A decreasing trend would suggest that many defects make transport more difficult in this simplified model.

Together, these plots translate a numerical sweep into physical intuition.


## Part C: Defect Strength Sweep

In [ ]:
def sweep_defect_strength(
    n_sites=40,
    hopping=1.0,
    epsilon_0=0.0,
    num_defects=5,
    strength_values=None,
    seed=42,
):
    if strength_values is None:
        strength_values = np.linspace(0.1, 5.0, 30)

    rng = np.random.default_rng(seed)
    clean = build_clean_hamiltonian(n_sites, hopping, epsilon_0)
    fixed_sites = np.sort(rng.choice(n_sites, size=num_defects, replace=False))
    rows = []

    for strength in strength_values:
        defected = add_defects_at_sites(clean, fixed_sites, strength)
        results = analyze_hamiltonian(defected)
        rows.append({
            'experiment': 'defect_strength_sweep',
            'arrangement': 'fixed_random',
            'num_defects': num_defects,
            'defect_strength': float(strength),
            'defect_sites': ';'.join(map(str, fixed_sites.tolist())),
            'mean_ipr': results['mean_ipr'],
            'max_ipr': results['max_ipr'],
            'transport_score': results['transport_score'],
        })

    return pd.DataFrame(rows)


df_strength = sweep_defect_strength()
df_strength.head()

### Explanation of the code cell above

This cell defines and runs a sweep over defect strength.

The number and positions of defects are kept fixed, while the defect strength changes over a range of values.

This is important because it isolates one physical factor:

> What happens when the same defect pattern becomes weaker or stronger?

The output `df_strength` stores the descriptors for each defect-strength value.


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(df_strength['defect_strength'], df_strength['mean_ipr'], marker='o')
plt.xlabel('Defect strength')
plt.ylabel('Mean IPR')
plt.title('Effect of Defect Strength on Localization')
plt.grid(True)
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'defect_strength_vs_mean_ipr_day2.png', dpi=200)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(df_strength['defect_strength'], df_strength['transport_score'], marker='o')
plt.xlabel('Defect strength')
plt.ylabel('Transport score')
plt.title('Effect of Defect Strength on Transport Descriptor')
plt.grid(True)
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'defect_strength_vs_transport_day2.png', dpi=200)
plt.show()

### Explanation of the code cell above

This cell visualizes the defect-strength sweep.

The first plot shows how localization changes with defect strength. The second plot shows how the transport score changes with defect strength.

If stronger defects increase IPR and decrease transport score, that supports the interpretation that strong local perturbations trap the eigenstates more strongly.


## Part D: Defect Arrangement Comparison

In [ ]:
def generate_defect_sites(n_sites, num_defects, mode='random', seed=42):
    rng = np.random.default_rng(seed)

    if mode == 'random':
        return np.sort(rng.choice(n_sites, size=num_defects, replace=False))
    if mode == 'clustered':
        start = int(rng.integers(0, n_sites - num_defects + 1))
        return np.arange(start, start + num_defects)
    if mode == 'spread':
        return np.linspace(0, n_sites - 1, num_defects, dtype=int)

    raise ValueError("mode must be one of: 'random', 'clustered', 'spread'")


def compare_defect_arrangements(
    n_sites=40,
    hopping=1.0,
    epsilon_0=0.0,
    num_defects=5,
    defect_strength=2.0,
    seed=42,
):
    clean = build_clean_hamiltonian(n_sites, hopping, epsilon_0)
    rows = []

    for mode in ['random', 'clustered', 'spread']:
        sites = generate_defect_sites(n_sites, num_defects, mode=mode, seed=seed)
        defected = add_defects_at_sites(clean, sites, defect_strength)
        results = analyze_hamiltonian(defected)
        rows.append({
            'experiment': 'defect_arrangement_comparison',
            'arrangement': mode,
            'num_defects': num_defects,
            'defect_strength': defect_strength,
            'defect_sites': ';'.join(map(str, sites.tolist())),
            'mean_ipr': results['mean_ipr'],
            'max_ipr': results['max_ipr'],
            'transport_score': results['transport_score'],
        })

    return pd.DataFrame(rows)


df_arrangement = compare_defect_arrangements()
df_arrangement

### Explanation of the code cell above

This cell compares three spatial arrangements of defects.

- `random`: defects are placed randomly.
- `clustered`: defects are placed next to each other.
- `spread`: defects are distributed across the lattice.

The function `compare_defect_arrangements` evaluates each arrangement using the same number of defects and the same defect strength. This makes the comparison fair.

The main scientific question is whether location pattern matters, not only defect count or strength.


In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(df_arrangement['arrangement'], df_arrangement['mean_ipr'])
plt.xlabel('Defect arrangement')
plt.ylabel('Mean IPR')
plt.title('Effect of Defect Arrangement on Localization')
plt.grid(axis='y')
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'arrangement_vs_mean_ipr_day2.png', dpi=200)
plt.show()

plt.figure(figsize=(8, 5))
plt.bar(df_arrangement['arrangement'], df_arrangement['transport_score'])
plt.xlabel('Defect arrangement')
plt.ylabel('Transport score')
plt.title('Effect of Defect Arrangement on Transport Descriptor')
plt.grid(axis='y')
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'arrangement_vs_transport_day2.png', dpi=200)
plt.show()

### Explanation of the code cell above

This cell plots the effect of defect arrangement.

The mean IPR plot shows which arrangement creates stronger localization. The transport-score plot shows which arrangement is more favorable for transport.

For example, if clustered defects give higher IPR, this may indicate that local defect groups trap states more effectively than spread defects.


## Part E: Generate the Day 2 Dataset

In [ ]:
day2_dataset = pd.concat(
    [df_number, df_strength, df_arrangement],
    ignore_index=True,
)

dataset_path = DATA_DIR / 'day2_defect_configuration_dataset.csv'
day2_dataset.to_csv(dataset_path, index=False)

print(f'Saved: {dataset_path}')
day2_dataset.head()

### Explanation of the code cell above

This cell combines all Day 2 experiments into one dataset.

It concatenates:

- the defect-number sweep,
- the defect-strength sweep,
- the arrangement comparison.

The combined dataset is saved as:

`day2_defect_configuration_dataset.csv`

This file is useful for reporting, later analysis, or as a small structured dataset for future ML experiments.


## Day 2 Conclusion

In this notebook, we extended the Day 1 lattice Hamiltonian model by systematically analyzing hydrogen-induced defect configurations. We compared clean and defected spectra, measured spectral shifts, and studied how defect number, defect strength, and spatial arrangement influence localization and the transport descriptor. The results provide a more controlled physics-informed dataset for later machine learning and quantum machine learning analysis.

## Key learning takeaway

After running this notebook, focus on writing one or two sentences in your own words:

- What physical quantity was computed?
- Which feature or parameter changed?
- How did the result affect localization or transport?

This habit will help you turn code execution into scientific understanding.


## Final Formula Reminder

The core logic of this notebook can be summarized as

$$
\text{defect configuration} \rightarrow H \rightarrow \{E_n,\mathbf{c}^{(n)}\} \rightarrow \text{features} \rightarrow \text{prediction or optimization}.
$$

This is the main physics-informed workflow: we start from a material-inspired defect model, convert it into spectral and localization descriptors, and then use those descriptors for machine learning, interpretation, or optimization.
